In [1]:
import re, json
import pandas as pd
import matplotlib.pyplot as plt

cw = pd.read_json("../logs-reels/cowrie_sample.json", lines=True)
cw["timestamp"] = pd.to_datetime(cw["timestamp"], utc=True)
print(cw.shape)
print(cw["eventid"].value_counts())

(10, 14)
eventid
cowrie.command.input    7
cowrie.login.failed     2
cowrie.login.success    1
Name: count, dtype: int64


In [2]:
regles = [
    (r"\buname\b|/proc/cpuinfo|\blscpu\b|\bhostname\b|/etc/os-release", "Reconnaissance système",   "T1082"),
    (r"\bwhoami\b|\bid\b|\bwho\b|\bw\b|\busers\b",                      "Découverte utilisateur",   "T1033"),
    (r"/etc/passwd|/etc/shadow|\bcat /etc/",                            "Découverte de comptes",    "T1087.001"),
    (r"\bps\b|\btop\b",                                                 "Découverte de processus",  "T1057"),
    (r"wget|curl|tftp|ftpget",                                          "Téléchargement de charge", "T1105"),
    (r"chmod \+x|\./|sh |bash ",                                        "Exécution de script",      "T1059.004"),
    (r"crontab|authorized_keys|\.ssh",                                  "Persistance",              "T1053 / T1098"),
    (r"rm -rf|history -c|unset HISTFILE",                               "Effacement de traces",     "T1070"),
    (r"xmrig|minerd|stratum|cpuminer",                                  "Minage de crypto",         "T1496"),
]

def classer(cmd):
    return [(i, t) for motif, i, t in regles if re.search(motif, cmd)]
    

In [3]:
cmds = cw[cw["eventid"] == "cowrie.command.input"].copy()
cmds["classification"] = cmds["input"].apply(classer)

# commandes non classées : à vérifier à la main
print("Non classées :", cmds[cmds["classification"].str.len() == 0]["input"].tolist())

detail = cmds.explode("classification").dropna(subset=["classification"])
detail["intention"] = detail["classification"].str[0]
detail["technique"] = detail["classification"].str[1]

synthese = (detail.groupby(["intention", "technique"]).size()
            .reset_index(name="occurrences")
            .sort_values("occurrences", ascending=False))
synthese


Non classées : []


,intention,technique,occurrences
0,Découverte de comptes,T1087.001,2
4,Téléchargement de charge,T1105,2
1,Découverte de processus,T1057,1
2,Découverte utilisateur,T1033,1
3,Reconnaissance système,T1082,1


In [4]:
lignes = []
with open("../logs-reels/opencanary_sample.log") as f:
    for l in f:
        d = json.loads(l)
        if d["logtype"] == 1001:        # messages de démarrage, pas des attaques
            continue
        lignes.append({
            "timestamp": pd.to_datetime(d["utc_time"], utc=True),
            "src_ip": d["src_host"],
            "dst_port": d["dst_port"],
            "logtype": d["logtype"],
            "username": d["logdata"].get("USERNAME"),
            "password": d["logdata"].get("PASSWORD"),
            "useragent": d["logdata"].get("USERAGENT"),
        })

oc = pd.DataFrame(lignes)
oc["service"] = oc["dst_port"].map({21: "FTP", 80: "HTTP", 3306: "MySQL", 445: "SMB"}).fillna("autre")
oc

,timestamp,src_ip,dst_port,logtype,username,password,useragent,service
0,2026-11-18 12:30:44.436918+00:00,127.0.0.1,21,2000,admin,test,NaN,FTP
1,2026-11-18 12:30:44.456796+00:00,127.0.0.1,80,3001,admin,test,curl/8.20.0,HTTP
2,2026-11-18 12:30:44.470797+00:00,127.0.0.1,80,3001,root,123456,curl/8.20.0,HTTP


In [5]:
a = cw[cw["eventid"].isin(["cowrie.login.failed", "cowrie.login.success"])][
        ["timestamp", "src_ip", "dst_port", "username", "password"]].copy()
a["service"] = "SSH"
a["succes"] = cw.loc[a.index, "eventid"].eq("cowrie.login.success")

b = oc[["timestamp", "src_ip", "dst_port", "username", "password", "service"]].copy()
b["succes"] = False   # OpenCanary ne dit pas si le login a réussi

connexions = pd.concat([a, b], ignore_index=True).sort_values("timestamp")
connexions

,timestamp,src_ip,dst_port,username,password,service,succes
3,2026-11-18 12:30:44.436918+00:00,127.0.0.1,21,admin,test,FTP,False
4,2026-11-18 12:30:44.456796+00:00,127.0.0.1,80,admin,test,HTTP,False
5,2026-11-18 12:30:44.470797+00:00,127.0.0.1,80,root,123456,HTTP,False
0,2026-11-18 12:33:05.435250+00:00,172.20.10.2,2222,root,"&\xc3\x89""'(-",SSH,False
1,2026-11-18 12:33:15.837006+00:00,172.20.10.2,2222,root,password*,SSH,False
2,2026-11-18 12:33:21.681999+00:00,172.20.10.2,2222,root,password,SSH,True


In [6]:
synthese.to_csv("../preuves/synthese-mitre-reel.csv", index=False)
detail[["session", "src_ip", "timestamp", "input", "intention", "technique"]].to_csv(
    "../preuves/commandes-classees-reel.csv", index=False)
connexions.to_csv("../preuves/connexions-reel.csv", index=False)